# 06. Comprehensive Visualizations for Thesis

**Objective**: Create publication-ready visualizations for thesis

This notebook generates:
1. Descriptive statistics plots
2. Relationship scatter plots
3. Time series trends
4. Regression results visualizations
5. Model comparison charts
6. Entity-specific analysis

All plots are designed for thesis inclusion with:
- High resolution (300 DPI)
- Clear labels and titles
- Professional styling
- Appropriate color schemes

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path
import json

# Add src to path
sys.path.append('..')

from src.data.loader import DataLoader
from src.utils.config import ConfigManager
from src.visualization.plots import RegressionPlotter

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

# Set high-quality plot style for thesis
plt.rcParams['figure.dpi'] = 100  # Display resolution
plt.rcParams['savefig.dpi'] = 300  # Save resolution
plt.rcParams['font.size'] = 11
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.style.use('seaborn-v0_8-whitegrid')

# Color palette
colors = sns.color_palette('Set2', 8)
sns.set_palette(colors)

print("Libraries imported successfully!")
print("Plot settings configured for high-quality thesis figures")

## 1. Load Data and Results

**Important**: Results are from Fixed Effects model with **entity-clustered standard errors** to account for autocorrelation in panel data.

In [ ]:
# Load configuration
config_manager = ConfigManager(config_dir='../config')
variables = config_manager.get_variable_info()

# Load data
loader = DataLoader()
data = loader.load_json('../data/processed/audit_data.json')

# Load regression results
with open('../results/final_panel_model_results.json', 'r') as f:
    results = json.load(f)

# Get variable names
entity_col = variables['panel_structure']['entity_variable']
time_col = variables['panel_structure']['time_variable']
dep_var = variables['dependent_variable']['name']
indep_vars = [v['name'] for v in variables['independent_variables']]
control_vars = [v['name'] for v in variables['control_variables']]
all_vars = [dep_var] + indep_vars + control_vars

print(f"Data loaded: {len(data)} observations")
print(f"Model type: {results['model_type']}")
print(f"R²: {results['r_squared']:.4f}")

## 2. Figure 1: Variable Distributions (Panel of Histograms)

Show the distribution of all key variables.

In [ ]:
# Create figure with subplots
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, var in enumerate(all_vars):
    # Histogram with KDE
    axes[i].hist(data[var], bins=25, alpha=0.7, color=colors[i], edgecolor='black', density=True)
    
    # Add KDE curve
    from scipy import stats as sp_stats
    kde_data = data[var].dropna()
    if len(kde_data) > 0:
        kde = sp_stats.gaussian_kde(kde_data)
        x_range = np.linspace(kde_data.min(), kde_data.max(), 100)
        axes[i].plot(x_range, kde(x_range), 'k-', linewidth=2, label='KDE')
    
    # Add mean and median lines
    axes[i].axvline(data[var].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {data[var].mean():.2f}')
    axes[i].axvline(data[var].median(), color='blue', linestyle='-.', linewidth=2, label=f'Median: {data[var].median():.2f}')
    
    # Labels
    axes[i].set_xlabel(var, fontweight='bold')
    axes[i].set_ylabel('Density', fontweight='bold')
    axes[i].set_title(f'Distribution of {var}', fontweight='bold', fontsize=12)
    axes[i].legend(loc='upper right', fontsize=8)
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Figure 1: Distribution of Variables', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('../results/figures/figure1_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Figure 1 saved: results/figures/figure1_distributions.png")

## 3. Figure 2: Bivariate Relationships

Scatter plots showing relationship between independent variables and dependent variable.

In [ ]:
# Scatter plots with regression lines
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, var in enumerate(indep_vars):
    # Scatter plot
    axes[i].scatter(data[var], data[dep_var], alpha=0.5, s=50, color=colors[i])
    
    # Add regression line
    from scipy.stats import linregress
    slope, intercept, r_value, p_value, std_err = linregress(data[var].dropna(), 
                                                               data[dep_var][data[var].notna()])
    x_line = np.array([data[var].min(), data[var].max()])
    y_line = slope * x_line + intercept
    axes[i].plot(x_line, y_line, 'r-', linewidth=2, 
                 label=f'R² = {r_value**2:.3f}\np = {p_value:.4f}')
    
    # Labels
    axes[i].set_xlabel(var, fontweight='bold')
    axes[i].set_ylabel(dep_var, fontweight='bold')
    axes[i].set_title(f'{dep_var} vs {var}', fontweight='bold')
    axes[i].legend(loc='best')
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Figure 2: Bivariate Relationships', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/figure2_bivariate.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Figure 2 saved: results/figures/figure2_bivariate.png")

## 4. Figure 3: Time Series Trends

Show how variables evolve over time (2021-2024).

In [ ]:
# Time trends with confidence bands
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, var in enumerate(all_vars):
    # Calculate yearly statistics
    yearly_mean = data.groupby(time_col)[var].mean()
    yearly_std = data.groupby(time_col)[var].std()
    yearly_sem = data.groupby(time_col)[var].sem()  # Standard error of mean
    
    # Plot line with confidence interval
    years = yearly_mean.index
    axes[i].plot(years, yearly_mean.values, marker='o', linewidth=2.5, 
                 markersize=10, color=colors[i], label='Mean')
    
    # 95% confidence interval
    ci = 1.96 * yearly_sem
    axes[i].fill_between(years, 
                          yearly_mean - ci, 
                          yearly_mean + ci, 
                          alpha=0.3, color=colors[i], label='95% CI')
    
    # Labels
    axes[i].set_xlabel('Year', fontweight='bold')
    axes[i].set_ylabel(var, fontweight='bold')
    axes[i].set_title(f'{var} Over Time', fontweight='bold')
    axes[i].legend(loc='best')
    axes[i].grid(True, alpha=0.3)
    axes[i].set_xticks(years)

plt.suptitle('Figure 3: Temporal Trends (2021-2024)', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('../results/figures/figure3_time_trends.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Figure 3 saved: results/figures/figure3_time_trends.png")

## 5. Figure 4: Correlation Heatmap

Professional correlation matrix for thesis.

In [ ]:
# Correlation matrix
corr_matrix = data[all_vars].corr()

# Create heatmap
fig, ax = plt.subplots(figsize=(10, 8))

# Mask upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# Plot heatmap
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f', 
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8},
            ax=ax)

ax.set_title('Figure 4: Correlation Matrix of Variables', 
             fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('../results/figures/figure4_correlation.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Figure 4 saved: results/figures/figure4_correlation.png")

## 6. Figure 5: Regression Coefficients with Confidence Intervals

Forest plot showing coefficient estimates from panel model.

In [ ]:
# Extract coefficients from results
coefs_dict = results['coefficients']
se_dict = results['std_errors']
pvals_dict = results['pvalues']

# Prepare data (exclude constant)
vars_plot = [v for v in indep_vars + control_vars if v in coefs_dict]
coefs = [coefs_dict[v] for v in vars_plot]
ses = [se_dict[v] for v in vars_plot]
pvals = [pvals_dict[v] for v in vars_plot]

# Calculate confidence intervals
ci_lower = [c - 1.96*s for c, s in zip(coefs, ses)]
ci_upper = [c + 1.96*s for c, s in zip(coefs, ses)]

# Create forest plot
fig, ax = plt.subplots(figsize=(10, 6))

y_pos = np.arange(len(vars_plot))

# Color by significance
colors_sig = ['green' if p < 0.05 else 'gray' for p in pvals]

# Plot points and error bars
for i, (y, coef, ci_l, ci_u, col) in enumerate(zip(y_pos, coefs, ci_lower, ci_upper, colors_sig)):
    ax.plot([ci_l, ci_u], [y, y], color=col, linewidth=2)
    ax.plot(coef, y, 'o', markersize=10, color=col, 
            label='Significant (p<0.05)' if col == 'green' and i == 0 else '')

# Add zero line
ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='No effect')

# Labels
ax.set_yticks(y_pos)
ax.set_yticklabels(vars_plot)
ax.set_xlabel('Coefficient Estimate', fontweight='bold', fontsize=12)
ax.set_title(f'Figure 5: Regression Coefficients with 95% Confidence Intervals\n({results["model_type"]} with Entity-Clustered SEs)', 
             fontweight='bold', fontsize=14)
ax.legend(loc='best')
ax.grid(True, alpha=0.3, axis='x')

# Add note
fig.text(0.5, 0.02, 'Note: Standard errors clustered by company to account for autocorrelation', 
         ha='center', fontsize=9, style='italic')

plt.tight_layout()
plt.savefig('../results/figures/figure5_coefficients.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Figure 5 saved: results/figures/figure5_coefficients.png")

## 7. Figure 6: Hypothesis Testing Summary

Visual summary of hypothesis testing results.

In [ ]:
# Extract hypothesis results
hyp_results = results['hypothesis_results']
hyp_df = pd.DataFrame(hyp_results)

# Create visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Coefficient magnitudes
colors_decision = ['green' if 'Supported' in d and 'Not' not in d else 'red' 
                   for d in hyp_df['Decision']]
bars = ax1.barh(hyp_df['Hypothesis'], hyp_df['Coefficient'], color=colors_decision, alpha=0.7)
ax1.axvline(x=0, color='black', linestyle='-', linewidth=1)
ax1.set_xlabel('Coefficient Estimate', fontweight='bold')
ax1.set_title('Coefficient Estimates by Hypothesis', fontweight='bold')
ax1.grid(True, alpha=0.3, axis='x')

# Add value labels
for i, (bar, val) in enumerate(zip(bars, hyp_df['Coefficient'])):
    ax1.text(val, bar.get_y() + bar.get_height()/2, f'{val:.4f}', 
             va='center', ha='left' if val > 0 else 'right', fontweight='bold')

# Right: P-values
bars2 = ax2.barh(hyp_df['Hypothesis'], -np.log10(hyp_df['p_value']), 
                 color=colors_decision, alpha=0.7)
ax2.axvline(x=-np.log10(0.05), color='red', linestyle='--', linewidth=2, 
            label='α = 0.05')
ax2.axvline(x=-np.log10(0.01), color='orange', linestyle='--', linewidth=2, 
            label='α = 0.01')
ax2.set_xlabel('-log₁₀(p-value)', fontweight='bold')
ax2.set_title('Significance Levels', fontweight='bold')
ax2.legend(loc='best')
ax2.grid(True, alpha=0.3, axis='x')

plt.suptitle('Figure 6: Hypothesis Testing Results', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/figure6_hypothesis_testing.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Figure 6 saved: results/figures/figure6_hypothesis_testing.png")

## 8. Figure 7: Panel Variation Decomposition

Show between vs within variation for panel data.

In [ ]:
# Load variation decomposition results
variation_df = pd.read_csv('../results/panel_variation_decomposition.csv')

# Create stacked bar chart
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(variation_df))
width = 0.35

# Plot between and within variation
bars1 = ax.bar(x - width/2, variation_df['Between_SD'], width, 
               label='Between (Cross-sectional)', alpha=0.8, color='steelblue')
bars2 = ax.bar(x + width/2, variation_df['Within_SD'], width, 
               label='Within (Time-series)', alpha=0.8, color='coral')

# Labels
ax.set_xlabel('Variable', fontweight='bold', fontsize=12)
ax.set_ylabel('Standard Deviation', fontweight='bold', fontsize=12)
ax.set_title('Figure 7: Panel Data Variation Decomposition\n(Between vs Within Variation)', 
             fontweight='bold', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(variation_df['Variable'], rotation=45, ha='right')
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.2f}', ha='center', va='bottom', fontsize=8)

for bar in bars2:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.2f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('../results/figures/figure7_panel_variation.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Figure 7 saved: results/figures/figure7_panel_variation.png")

## 9. Figure 8: Entity Heterogeneity

Show variation across entities (companies) for key variables.

In [ ]:
# Calculate entity averages
entity_means = data.groupby(entity_col)[dep_var].mean().sort_values(ascending=False)

# Create figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left: Bar chart of entity means
top_n = min(15, len(entity_means))  # Show top 15 or all if less
entity_means_top = entity_means.head(top_n)

bars = ax1.barh(range(top_n), entity_means_top.values, alpha=0.7)
ax1.set_yticks(range(top_n))
ax1.set_yticklabels(entity_means_top.index)
ax1.set_xlabel(f'Average {dep_var}', fontweight='bold')
ax1.set_title(f'Top {top_n} Companies by Average {dep_var}', fontweight='bold')
ax1.grid(True, alpha=0.3, axis='x')

# Right: Box plot by year
data.boxplot(column=dep_var, by=time_col, ax=ax2)
ax2.set_xlabel('Year', fontweight='bold')
ax2.set_ylabel(dep_var, fontweight='bold')
ax2.set_title(f'{dep_var} Distribution by Year', fontweight='bold')
ax2.get_figure().suptitle('')  # Remove default title

plt.suptitle('Figure 8: Entity and Temporal Heterogeneity', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/figure8_heterogeneity.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Figure 8 saved: results/figures/figure8_heterogeneity.png")

## 10. Figure 9: Model Comparison Summary

Compare Pooled OLS vs Panel methods.

In [ ]:
# Load comparison results
try:
    comparison_df = pd.read_csv('../results/pooled_vs_panel_comparison.csv')
    
    # Extract numeric coefficients (remove significance stars)
    def extract_coef(s):
        import re
        match = re.search(r'(-?\d+\.\d+)', str(s))
        return float(match.group(1)) if match else np.nan
    
    pooled_coefs = [extract_coef(c) for c in comparison_df['Pooled_OLS']]
    fe_coefs = [extract_coef(c) for c in comparison_df['Fixed_Effects']]
    re_coefs = [extract_coef(c) for c in comparison_df['Random_Effects']]
    
    # Create grouped bar chart
    fig, ax = plt.subplots(figsize=(12, 6))
    
    x = np.arange(len(comparison_df))
    width = 0.25
    
    ax.bar(x - width, pooled_coefs, width, label='Pooled OLS', alpha=0.8, color='red')
    ax.bar(x, fe_coefs, width, label='Fixed Effects', alpha=0.8, color='green')
    ax.bar(x + width, re_coefs, width, label='Random Effects', alpha=0.8, color='blue')
    
    ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
    ax.set_xlabel('Variable', fontweight='bold', fontsize=12)
    ax.set_ylabel('Coefficient Estimate', fontweight='bold', fontsize=12)
    ax.set_title('Figure 9: Model Comparison - Coefficient Estimates', 
                 fontweight='bold', fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels(comparison_df['Variable'], rotation=45, ha='right')
    ax.legend(loc='best', fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('../results/figures/figure9_model_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ Figure 9 saved: results/figures/figure9_model_comparison.png")
    
except FileNotFoundError:
    print("⚠ Comparison file not found. Run notebook 05 first.")

## 11. Create Figure Summary Document

In [ ]:
# Create summary of all figures for thesis reference
figure_summary = """
FIGURE SUMMARY FOR THESIS
=========================

All figures saved in: results/figures/
Resolution: 300 DPI (publication quality)
Format: PNG with transparent background where applicable

IMPORTANT: Regression results (Figures 5 & 6) use Fixed Effects model with 
ENTITY-CLUSTERED STANDARD ERRORS to account for autocorrelation in panel data.

Figure 1: Variable Distributions
  - Shows distribution of all key variables with KDE curves
  - Includes mean and median lines
  - Use in: Descriptive Statistics section

Figure 2: Bivariate Relationships
  - Scatter plots with regression lines
  - Shows preliminary relationships between IVs and DV
  - Use in: Preliminary Analysis section

Figure 3: Temporal Trends (2021-2024)
  - Time series plots with confidence intervals
  - Shows evolution of variables over time
  - Use in: Descriptive Statistics section

Figure 4: Correlation Matrix
  - Heatmap of variable correlations
  - Lower triangle only (cleaner presentation)
  - Use in: Preliminary Analysis or Appendix

Figure 5: Regression Coefficients *** PRIMARY RESULTS FIGURE ***
  - Forest plot with 95% confidence intervals
  - Uses CLUSTERED STANDARD ERRORS (accounts for autocorrelation)
  - Main results visualization
  - Use in: Results section (PRIMARY FIGURE)
  - Caption should mention: "Standard errors clustered by company"

Figure 6: Hypothesis Testing Results
  - Dual panel: coefficients and significance
  - Uses CLUSTERED STANDARD ERRORS
  - Clear visualization of hypothesis decisions
  - Use in: Results section

Figure 7: Panel Variation Decomposition
  - Between vs within variation comparison
  - Justifies panel data approach
  - Use in: Methodology or Results section

Figure 8: Entity Heterogeneity
  - Shows cross-sectional and temporal variation
  - Box plots and entity rankings
  - Use in: Descriptive Statistics section

Figure 9: Model Comparison
  - Compares Pooled OLS vs Panel methods
  - Demonstrates need for panel approach
  - Use in: Methodology section

RECOMMENDED FIGURE SEQUENCE IN THESIS:
1. Figure 1 (Distributions) - Descriptive Statistics
2. Figure 8 (Heterogeneity) - Descriptive Statistics  
3. Figure 3 (Time Trends) - Descriptive Statistics
4. Figure 4 (Correlations) - Preliminary Analysis
5. Figure 7 (Variation) - Methodology Justification
6. Figure 9 (Model Comparison) - Methodology Justification
7. Figure 5 (Main Results) - Results Section *** MAIN FIGURE ***
8. Figure 6 (Hypothesis Testing) - Results Section
9. Figure 2 (Bivariate) - Appendix (optional)

FIGURE CAPTIONS - SUGGESTED TEXT:

Figure 5 Caption:
"Coefficient estimates from Fixed Effects regression with 95% confidence intervals.
Standard errors are clustered by company to account for within-firm correlation 
over time. Green markers indicate statistical significance at the 5% level."

Figure 6 Caption:
"Summary of hypothesis testing results. Left panel shows coefficient estimates 
for each hypothesis. Right panel shows significance levels (-log₁₀ p-value). 
Standard errors are clustered by company. Dashed lines indicate significance 
thresholds at α = 0.05 and α = 0.01."
"""

# Save summary
with open('../results/figures/FIGURE_SUMMARY.txt', 'w') as f:
    f.write(figure_summary)

print(figure_summary)
print("\n✓ Figure summary saved: results/figures/FIGURE_SUMMARY.txt")

## Summary

This notebook created comprehensive visualizations:
- ✓ 9 publication-ready figures at 300 DPI
- ✓ Professional styling for thesis inclusion
- ✓ Clear labels, titles, and legends
- ✓ Appropriate color schemes
- ✓ Figure summary document for reference

**All figures saved in**: `results/figures/`

**For Thesis**:
1. Use figures in the recommended sequence
2. Add figure captions explaining key findings
3. Reference figures in text by number
4. Figure 5 (Regression Coefficients) is the primary results figure

**Next Steps**:
1. Review all figures for thesis suitability
2. Add figure captions in thesis document
3. Customize colors/styles if needed for university requirements
4. Consider converting to other formats (PDF, EPS) if required